In [12]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [13]:
# Domain layer: Pure business logic for articles and summarization
from dataclasses import dataclass
from typing import Optional
import nltk
from nltk.tokenize import sent_tokenize

# Download NLTK resources (run once)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

@dataclass
class Article:
    """Represents a news article with cleaned and summarized content."""
    title: str
    summary: str
    link: str
    pub_date: str
    thumbnail: Optional[str]

def simple_summarize(text: str, max_sentences: int = 2) -> str:
    """Summarize text by extracting the first few sentences."""
    sentences = sent_tokenize(text)
    summary = ' '.join(sentences[:min(max_sentences, len(sentences))])
    return summary.strip()

In [14]:
# Infrastructure layer: Fetch and clean RSS feeds
import feedparser
from bs4 import BeautifulSoup
from typing import List, Optional
from /domain.article import Article

def clean_text(html_text: str) -> str:
    """Remove HTML tags and clean text."""
    if not html_text:
        return ""
    soup = BeautifulSoup(html_text, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    return text

def fetch_rss_feed(feed_url: str, max_articles: int = 5) -> List[Article]:
    """Fetch and parse an RSS feed into a list of Articles."""
    try:
        feed = feedparser.parse(feed_url)
        articles = []
        for entry in feed.entries[:max_articles]:
            title = entry.get('title', '')
            description = clean_text(entry.get('description', '') or entry.get('summary', ''))
            link = entry.get('link', '')
            pub_date = entry.get('published', '') or entry.get('updated', '')
            # Extract thumbnail (varies by feed)
            thumbnail = None
            if 'media_thumbnail' in entry:
                thumbnail = entry.media_thumbnail[0].get('url')
            elif 'media_content' in entry:
                for content in entry.media_content:
                    if content.get('medium') == 'image':
                        thumbnail = content.get('url')
                        break
            articles.append(Article(
                title=title,
                summary=description,  # Will be summarized later
                link=link,
                pub_date=pub_date,
                thumbnail=thumbnail
            ))
        return articles
    except Exception as e:
        print(f"Error fetching feed {feed_url}: {e}")
        return []

SyntaxError: invalid syntax (2470591742.py, line 5)

In [7]:
# Application layer:  RSS feed processing
from typing import List
from domain.article import Article, simple_summarize
from infrastructure.rss_repository import fetch_rss_feed

def process_rss_feed(feed_url: str, max_articles: int = 5) -> List[Article]:
    """Fetch, clean, and summarize articles from an RSS feed."""
    articles = fetch_rss_feed(feed_url, max_articles)
    # Summarize descriptions
    for article in articles:
        article.summary = simple_summarize(article.summary, max_sentences=2)
    return articles


## Testing step 3

In [8]:
# Set up project path
import sys
import os
sys.path.append(os.path.abspath('..'))

In [9]:
# Test RSS feed parsing
from application.feed_service import process_rss_feed

# Define feed URLs
feeds = {
    'skynews': 'https://feeds.skynews.com/feeds/rss/home.xml',
    'bbc': 'https://feeds.bbci.co.uk/news/rss.xml',
    'guardian': 'https://www.theguardian.com/world/rss'
}

# Test Sky News
articles = process_rss_feed(feeds['skynews'], max_articles=2)
for article in articles:
    print(f"Title: {article.title}")
    print(f"Summary: {article.summary}")
    print(f"Link: {article.link}")
    print(f"Pub Date: {article.pub_date}")
    print(f"Thumbnail: {article.thumbnail}")
    print("-" * 80)

Title: Hunting for Barbecue: Children going to school dodge gunfire as police battle gangs on crime-ridden streets
Summary: A group of school children in their smart uniforms skip past us, overseen by their mums and dads.
Link: https://news.sky.com/story/90-of-port-au-prince-controlled-by-gangs-as-thousands-forced-into-heaving-displacement-camps-13368885
Pub Date: Mon, 19 May 2025 10:00:00 +0100
Thumbnail: https://e3.365dm.com/25/05/1920x1080/skynews-police-haiti-port-au-prince_6918158.png?20250520073602
--------------------------------------------------------------------------------
Title: Girl, 11, unlawfully killed after she drowned at waterpark, inquest rules
Summary: A coroner has concluded that an 11-year-old girl was unlawfully killed after she drowned at a waterpark in Berkshire in 2022.
Link: https://news.sky.com/story/girl-11-unlawfully-killed-after-she-drowned-at-waterpark-inquest-rules-13371361
Pub Date: Tue, 20 May 2025 10:08:00 +0100
Thumbnail: https://e3.365dm.com/22/08/

In [6]:
# Test all feeds
for feed_name, feed_url in feeds.items():
    print(f"\nTesting {feed_name.upper()}:")
    articles = process_rss_feed(feed_url, max_articles=2)
    print(f"Found {len(articles)} articles")
    for article in articles:
        print(f"Title: {article.title}")
        print(f"Summary: {article.summary}")
        print("-" * 80)


Testing SKYNEWS:
Found 2 articles
Title: Hunting for Barbecue: Children going to school dodge gunfire as police battle gangs on crime-ridden streets
Summary: A group of school children in their smart uniforms skip past us, overseen by their mums and dads.
--------------------------------------------------------------------------------
Title: Girl, 11, unlawfully killed after she drowned at waterpark, inquest rules
Summary: A coroner has concluded that an 11-year-old girl was unlawfully killed after she drowned at a waterpark in Berkshire in 2022.
--------------------------------------------------------------------------------

Testing BBC:
Found 2 articles
Title: Jeremy Bowen: Goodwill running out as allies demand that Israel end Gaza offensive
Summary: France, the UK and Canada are among nations that have sharpened their criticism of Israel's ground offensive.
--------------------------------------------------------------------------------
Title: Ministers consider easing winter fuel

In [10]:
# Test FastAPI API
import requests

# Test Sky News API
try:
    response = requests.get("http://127.0.0.1:8000/articles?feed=skynews&max_articles=2")
    response.raise_for_status()
    print("Sky News API Response:")
    for article in response.json():
        print(f"Title: {article['title']}")
        print(f"Summary: {article['summary']}")
        print(f"Link: {article['link']}")
        print(f"Pub Date: {article['pub_date']}")
        print(f"Thumbnail: {article['thumbnail']}")
        print("-" * 80)
except requests.RequestException as e:
    print(f"API Error: {e}")

# Test all feeds
feeds = ['skynews', 'bbc', 'guardian']
for feed in feeds:
    try:
        response = requests.get(f"http://127.0.0.1:8000/articles?feed={feed}&max_articles=2")
        response.raise_for_status()
        print(f"\n{feed.upper()} API Response:")
        print(f"Found {len(response.json())} articles")
        for article in response.json():
            print(f"Title: {article['title']}")
            print(f"Summary: {article['summary']}")
            print("-" * 80)
    except requests.RequestException as e:
        print(f"{feed.upper()} API Error: {e}")

API Error: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /articles?feed=skynews&max_articles=2 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x2ccd05a90>: Failed to establish a new connection: [Errno 61] Connection refused'))
SKYNEWS API Error: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /articles?feed=skynews&max_articles=2 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x2ccd3a720>: Failed to establish a new connection: [Errno 61] Connection refused'))
BBC API Error: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /articles?feed=bbc&max_articles=2 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x2ccd3ad50>: Failed to establish a new connection: [Errno 61] Connection refused'))
GUARDIAN API Error: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /articles?feed=guardia